In [14]:
import os
from PIL import Image
import numpy as np
import random
import csv
from tqdm import tqdm
import zipfile
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset, random_split
import torchmetrics

In [15]:
seed = 42
torch.manual_seed(seed)

EVALUATION = True
root_dir = "/personal/mock" if not EVALUATION else "/bohr/train-duyz/v1"

# Data

In [16]:
class CustomDataset(Dataset):
    def __init__(self, data_dir, train_file, mode=None, transform=None):
        self.transform = transform  # Image transformation pipeline (e.g., resizing, normalization)
        self.images = []  # Stores loaded images (PIL Image objects)
        self.labels = []  # Stores corresponding labels (0 for non-grid, 1 for grid collage)
        self.mode = mode  # Operation mode: "train", "val", or "test"

        # Read and parse the training annotation file
        with open(train_file, "r", encoding="gbk") as file: 
            for i, line in enumerate(file):
                line = line.strip()  # Remove extra spaces/newlines
                if not line:  # Skip empty lines
                    continue
                    
                # Split line into fields (columns)
                fields = [field.strip() for field in line.split(',')]
                
                # Validate field count for training mode
                if len(fields) != 4 and self.mode == "train":
                    raise ValueError("Each line must have 4 fields: id,img_url,label,class")
                
                # Process training data (includes labels)
                if self.mode == "train":
                    img_name, img_url, label_str, class_name = fields
                    label = int(label_str)  # Convert label from string to integer
                    
                    # Load image from local file (supports JPG and PNG)
                    try:
                        img_path = os.path.join(data_dir, f'{img_name}.jpg')
                        image = Image.open(img_path).convert('RGB')  # Ensure 3-channel RGB
                    except:  # If JPG not found, try PNG
                        img_path = os.path.join(data_dir, f'{img_name}.png')
                        image = Image.open(img_path).convert('RGB')

                    self.images.append(image)
                    self.labels.append(label)
                
                # Process validation/test data (no labels)
                elif self.mode == "val" or self.mode == "test":
                    img_name, img_url, class_name = fields  # No label in val/test files
                    
                    # Load image (same as training)
                    try:
                        img_path = os.path.join(data_dir, f'{img_name}.jpg')
                        image = Image.open(img_path).convert('RGB')
                    except:
                        img_path = os.path.join(data_dir, f'{img_name}.png')
                        image = Image.open(img_path).convert('RGB')

                    self.images.append(image)  # Only store images, no labels
                else:
                    raise ValueError("Invalid mode: must be 'train', 'val', or 'test'")
        
        ######################## Data Augmentation ###########################
        '''
        Data augmentation function: generates synthetic grid collages from non-grid images
        This helps balance the dataset (often fewer real grid collages)
        '''
        def augmentation(images, labels, augmentation_sample_num=400, COL=2, ROW=2):
            # Collect all non-grid images (label=0) to use for creating synthetic grids
            neg_images = []
            for (img, label) in zip(images, labels):
                # if label == 0:
                neg_images.append(img)
            
            # Shuffle the non-grid images to ensure randomness
            random.shuffle(neg_images)

            # Lists to store augmented images and their labels
            augmentation_images, augmentation_labels = [], []
            
            # Grid configuration: create 2x2 grids
            # COL = 2  # Number of columns in the grid
            # ROW = 2  # Number of rows in the grid
            UNIT_HEIGHT_SIZE = 128  # Height of each small image in the grid
            UNIT_WIDTH_SIZE = 128   # Width of each small image in the grid
            
            # Generate synthetic grid collages
            # Iterate over non-grid images in steps of 4 (since 2x2 grid needs 4 images)
            for i in range(0, len(neg_images)//(COL*ROW), COL*ROW):
                # Create a blank canvas for the grid collage
                augmentation_image = Image.new(
                    'RGB', 
                    (UNIT_WIDTH_SIZE * COL, UNIT_HEIGHT_SIZE * ROW)  # Total size: 256x256
                )
                
                # Paste each small image into its position in the grid
                for row in range(ROW):
                    for col in range(COL):
                        # Calculate index of the image to paste
                        img_index = i * COL * ROW + COL * row + col
                        # Paste the image at (column*width, row*height)
                        augmentation_image.paste(
                            neg_images[img_index].resize((UNIT_WIDTH_SIZE, UNIT_HEIGHT_SIZE), Image.BILINEAR),
                            (UNIT_WIDTH_SIZE * col, UNIT_HEIGHT_SIZE * row)
                        )
                
                # Add the synthetic grid collage to the dataset
                augmentation_images.append(augmentation_image)
                augmentation_labels.append(1)  # Label synthetic grids as 1

            return augmentation_images, augmentation_labels


        ######################## Apply Augmentation #############################
        # Only augment data during training
        if self.mode == "train":
            for r,c in [(1,2),(2,3),(2,2)]:
                # Generate augmented images and labels
                augmentation_images, augmentation_labels = augmentation(self.images, self.labels, 400, r, c)
                # Add augmented data to the original dataset
                self.images.extend(augmentation_images)
                self.labels.extend(augmentation_labels)

        self.labels = torch.tensor(self.labels)
    
    # Return the total number of samples in the dataset
    def __len__(self):
        return len(self.images)

    # Retrieve a single sample by index
    def __getitem__(self, idx):
        image = self.images[idx]  # Get the image at position 'idx'
        
        # Apply transformations if specified (e.g., resizing, normalization)
        if self.transform:
            image = self.transform(image)
            
        # For training, return both image and label; for val/test, return only image
        if self.mode == "train":
            label = self.labels[idx]
            return image, 1 if label >= 0.5 else 0
        else:
            return image

    def update(self, model, a=0.05):
        res = []
        for img, _ in self:
            res.append(img)

        with torch.no_grad():
            out = model(torch.stack(res).to("cuda")).cpu().squeeze(1)
            self.labels = self.labels * torch.tensor(1-a) + out * a

In [17]:
# 1. Define a stronger augmentation pipeline for training
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2,
                           contrast=0.2,
                           saturation=0.2,
                           hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

# Simple resize+normalize for validation / testing
transform_eval = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

# Load the full dataset (real images=0, synthetic grids=1)
full_dataset = CustomDataset(
    f"{root_dir}/train",
    f"{root_dir}/train.csv",
    mode="train",
    transform=transform_train,
)

# 2. Randomly split 80% train / 20% val
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_ds, val_ds = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Override the transform on the validation subset
# (they both point at the same underlying dataset object)
val_ds.dataset.transform = transform_eval

# Create DataLoaders
train_loader = DataLoader(
    train_ds, batch_size=64, shuffle=True, num_workers=4
)
val_loader   = DataLoader(
    val_ds,   batch_size=64, shuffle=False, num_workers=4
)

# Initialize training parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
print(f'Using device: {device}')

In [18]:
# sanity check
batch = next(iter(train_loader))
[b.shape for b in batch]

# Model

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        # Define the sequential model
        self.features = nn.Sequential(
            # Convolutional layer 1: 3 input channels (RGB), 8 output channels, 5x5 kernel
            nn.Conv2d(3, 8, 5),
            nn.ReLU(),
            # Max pooling layer: 2x2 kernel with stride 2 (halves size)
            nn.MaxPool2d(2, 2),
            
            # Convolutional layer 2: 8 input channels, 16 output channels, 5x5 kernel
            nn.BatchNorm2d(8),
            nn.Conv2d(8, 16, 5),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        # Fully connected layers
        self.classifier = nn.Sequential(
            nn.Linear(16 * 61 * 61, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Forward pass through first convolutional block: conv1 → ReLU → pool
        if x.dim()==3:
            x=x.unsqueeze(0)
            Squeeze=True
        else:
            Squeeze=False
        x = self.features(x)
        # print(x.shape)
        x = x.view(-1, 16 * 61 * 61)
        x = self.classifier(x)
        if Squeeze==True:
            x=x.squeeze(0)
        return x

In [20]:
model = MyModel()

model = model.to(device)
model(batch[0].to(device)).shape

# Training

In [21]:
# Training function to optimize the model parameters
def train_model(model, train_loader, criterion, optimizer, device, num_epochs=10):
    accuracy = torchmetrics.Accuracy(task='binary').to(device)
    max_accuracy = 0  # Track the highest validation accuracy during training
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for (inputs, labels) in tqdm(train_loader):
            inputs = inputs.to(device)
            labels = labels.to(device).float().view(-1, 1)  # Reshape to [batch_size, 1]
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
        model.eval()
        accuracy.reset()
        for inputs, labels in tqdm(val_loader):
            inputs, labels = inputs.to(device), labels.to(device).float().view(-1, 1)

            with torch.no_grad():
                outputs = model(inputs)
                accuracy(outputs, labels)

        epoch_loss = running_loss / len(train_loader.dataset)
        val_accuracy = accuracy.compute()*100

        log_message = (
            f"Epoch {epoch+1}/{num_epochs}, "
            f"Train Loss: {epoch_loss:.4f}, "
            f"Val Accuracy: {val_accuracy:.4f}%"
        )
        print(log_message)
        
        if val_accuracy > max_accuracy:
            max_accuracy = val_accuracy

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
epochs = 7

In [ ]:
for i in range(4):
    print(f"====== Iteration {i} ======")
    model.train()
    train_model(model, train_loader, criterion, optimizer, device, num_epochs=epochs)
    model.eval()
    full_dataset.update(model)

# Submission

In [ ]:
# Define prediction function for generating submission results
def predict(model, loader, device):
    """
    Generate predictions for a dataset using the trained model.
    
    Args:
        model (nn.Module): Trained model for inference.
        loader (DataLoader): DataLoader containing test samples.
        device (torch.device): Device to run inference on (e.g., 'cuda' or 'cpu').
        
    Returns:
        list: Predicted labels (0 or 1) for each sample in the dataset.
    """
    # Set model to evaluation mode (disables dropout/batchnorm)
    model.eval()
    
    # List to store predictions
    preds = []
    
    # Disable gradient computation for faster inference
    with torch.no_grad():
        # Iterate over batches with progress bar (tqdm)
        for batch in tqdm(loader, desc='Predicting'):
            # Move batch to device
            x = batch.to(device)
            
            output = model(x)
            pred = (output >= 0.5).int()
            
            # Collect predictions and move to CPU
            preds.extend(pred.cpu().numpy())
    
    return preds

In [ ]:
# Save prediction results to CSV format for competition submission
def save_submission_csv(preds, save_name):
    """
    Convert predictions to CSV format required by the competition.
    
    Args:
        preds (list): List of predicted labels (0 or 1).
        save_name (str): Output file name (e.g., "submission.csv").
        
    Format Requirements:
        - Single column with no header
        - No index column
        - Each row contains one prediction (0 or 1)
    """
    # Convert list of predictions to pandas DataFrame
    df = pd.DataFrame(preds)
    
    # Save to CSV with no index and no header (strict competition format)
    df.to_csv(save_name, index=False, header=False)

In [ ]:
if EVALUATION:
    # Get data path from environment variable (provided in competition system)
    DATA_PATH = os.environ.get("DATA_PATH") + "/" 

    # Define paths for validation and test sets
    val_dir = DATA_PATH + '/val'       # Validation set directory
    val_file = DATA_PATH + '/val.csv'   # Validation set annotations
    test_dir = DATA_PATH + '/test'      # Test set directory
    test_file = DATA_PATH + '/test.csv' # Test set annotations

    # Load validation set (public score - Leaderboard A)
    val_dataset = CustomDataset(val_dir, val_file, mode="val", transform=transform_eval)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)  # Shuffle not critical for inference

    # Load test set (private score - Leaderboard B)
    test_dataset = CustomDataset(test_dir, test_file, mode="test", transform=transform_eval)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)  # Keep order for submission

    # Generate predictions using the trained model
    val_preds = predict(model, val_loader, device)  # Predictions for Leaderboard A
    test_preds = predict(model, test_loader, device)  # Predictions for Leaderboard B

    # Generate submission files
    # Save validation predictions (Leaderboard A)
    save_submission_csv(val_preds, 'submissionA.csv')
    # Save test predictions (Leaderboard B)
    save_submission_csv(test_preds, 'submissionB.csv')

    # Package into zip for submission
    with zipfile.ZipFile('submission.zip', 'w') as zipf:
        zipf.write('submissionA.csv')
        zipf.write('submissionB.csv')

    # Clean up temporary files
    os.remove('submissionA.csv')
    os.remove('submissionB.csv')